---
title: "Lab 8: Linear Classifiers"
format: 
  html:
    embed-resources: true
execute:
  echo: true
code-fold: true
author: James Compagno
jupyter: python3
---

# The Data
This week, we consider a dataset generated from text data.

The original dataset can be found here: https://www.kaggle.com/datasets/kingburrito666/cannabis-strains. It consists of user reviews of different strains of cannabis. Users rated their experience with the cannabis strain on a scale of 1 to 5. They also selected words from a long list to describe the Effects and the Flavor of the cannabis.

In the dataset linked above, each row is one strain of cannabis. The average rating of all testers is reported, as well as the most commonly used words for the effect and flavor.

Some data cleaning has been performed for you: The Effect and Flavor columns have been converted to dummy variables indicating if the particular word was used for the particular strain.

This cleaned data can be found at: https://www.dropbox.com/s/s2a1uoiegitupjc/cannabis_full.csv
Our goal will be to fit models that identify the Sativa types from the Indica types, and then to fit models that also distinguish the Hybrid types.

IMPORTANT: In this assignment, you do not need to consider different feature sets. Normally, this would be a good thing to try - but for this homework, simply include all the predictors for every model.


In [9]:
import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, 
                             precision_recall_fscore_support, roc_auc_score, 
                             confusion_matrix, classification_report, roc_curve, 
                             auc, precision_score, recall_score, f1_score)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.svm import SVC


# Part One: Binary Classification

Create a dataset that is limited only to the Sativa and Indica type cannabis strains.

This section asks you to create a final best model for each of the four new model types studied this week: LDA, QDA, SVC, and SVM. For SVM, you may limit yourself to only the polynomial kernel.

For each, you should:

    - Choose a metric you will use to select your model, and briefly justify your choice. (Hint: There is no specific target category here, so this should not be a metric that only prioritizes one category.)

    - Find the best model for predicting the Type variable. Don't forget to tune any hyperparameters. 

    - Report the (cross-validated!) metric.
    
    - Fit the final model.
    
    - Output a confusion matrix.

For my metric I will choose ROC-AUC. 

In [10]:
weed = pd.read_csv("https://www.dropbox.com/s/s2a1uoiegitupjc/cannabis_full.csv?dl=1")
weed = weed.dropna()

weed.describe()

,Rating,Creative,Energetic,Tingly,Euphoric,Relaxed,Aroused,Happy,Uplifted,Hungry,Talkative,Giggly,Focused,Sleepy,Dry,Mouth,Earthy,Sweet,Citrus,Flowery,Violet,Diesel,Spicy/Herbal,Sage,Woody,Apricot,Grapefruit,Orange,Pungent,Grape,Pine,Skunk,Berry,Pepper,Menthol,Blue,Cheese,Chemical,Mango,Lemon,Peach,Vanilla,Nutty,Chestnut,Tea,Tobacco,Tropical,Strawberry,Blueberry,Mint,Apple,Honey,Lavender,Lime,Coffee,Ammonia,Minty,Tree,Fruit,Butter,Pineapple,Tar,Rose,Plum,Pear
count,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000
mean,4.430169,0.329986,0.283432,0.151073,0.727522,0.772250,0.088088,0.835691,0.670927,0.208581,0.158832,0.130534,0.264263,0.327704,0.000456,0.000456,0.504336,0.480146,0.240073,0.121406,0.003195,0.109539,0.102693,0.017800,0.116385,0.004108,0.017344,0.035144,0.205842,0.074395,0.154267,0.079416,0.162026,0.026472,0.010497,0.069375,0.029210,0.016887,0.014605,0.086262,0.002282,0.015518,0.011410,0.003195,0.007759,0.004108,0.069831,0.021451,0.066180,0.024646,0.007303,0.014149,0.016887,0.024190,0.010954,0.012780,0.018713,0.015518,0.015518,0.008672,0.019169,0.003651,0.007303,0.000913,0.001369
std,0.419576,0.470315,0.450767,0.358201,0.445336,0.419476,0.283487,0.370640,0.469984,0.406387,0.365602,0.336967,0.441041,0.469484,0.021364,0.021364,0.500095,0.499720,0.427225,0.326673,0.056446,0.312386,0.303627,0.132254,0.320759,0.063974,0.130578,0.184185,0.404408,0.262473,0.361287,0.270448,0.368559,0.160571,0.101941,0.254148,0.168434,0.128878,0.119994,0.280815,0.047727,0.123629,0.106232,0.056446,0.087763,0.063974,0.254920,0.144917,0.248653,0.155080,0.085162,0.118131,0.128878,0.153673,0.104110,0.112348,0.135540,0.123629,0.123629,0.092739,0.137151,0.060329,0.085162,0.030206,0.036986
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.300000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4.400000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.

In [11]:
for col in weed.columns:
    if col not in ['Type', 'Strain', 'Effects', 'Flavor']:
        weed[col] = pd.to_numeric(weed[col], errors='coerce')

# Separate X and Y
y = weed['Type']
X = weed.drop(columns=['Type', 'Strain', 'Effects', 'Flavor'])

# Binary Split 
binary_weed = weed['Type'].isin(['indica', 'sativa'])
X_binary = X[binary_weed] 
y_binary = y[binary_weed]

# Model Library 
model_library = {}
records = []

## Q1: LDA - Linear Discriminant Analysis

In [12]:
model_name = "LDA_Binary"
lda_model = LinearDiscriminantAnalysis()

# Cross-validated prediction
y_pred_cv = cross_val_predict(lda_model, X_binary, y_binary, cv=5)
y_proba_cv = cross_val_predict(lda_model, X_binary, y_binary, cv=5, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_binary, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_binary, y_proba_cv)
cv_accuracy = accuracy_score(y_binary, y_pred_cv)
precision = precision_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Fit 
lda_model.fit(X_binary, y_binary)

# Store in model library
model_library[model_name] = lda_model

# Store results 
records.append({
    "Model": model_name,
    "Classification": "Binary",
    "Classification Type": "LDA",
    "Hyperparameter 1 Name": "NA", 
    "Hyperparameter 1 Value": "NA",
    "Hyperparameter 2 Name": "NA", 
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": "NA",
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity,
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[597  62]
 [ 88 321]]


The LDA model is overall strong at identifying the correct strain with a ROC-AUC of 0.932. However, it had a lower Recall of (0.785) so some sativa strains were miss-classified. The model was very good however at classifying indica strains with a specificity of Specificity 0.906. 

## Q2: QDA - Quadratic Discriminant Analysis

In [13]:
model_name = "QDA_Binary"
qda_model = QuadraticDiscriminantAnalysis()

# Parameter grid for QDA 
param_grid = {
    'reg_param': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

# GridSearchCV
grid_search = GridSearchCV(
    qda_model,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_binary, y_binary)

# Best model
best_qda = grid_search.best_estimator_
best_reg_param = grid_search.best_params_['reg_param']

# Cross-validated prediction with best model
y_pred_cv = cross_val_predict(best_qda, X_binary, y_binary, cv=5)
y_proba_cv = cross_val_predict(best_qda, X_binary, y_binary, cv=5, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_binary, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_binary, y_proba_cv)
cv_accuracy = accuracy_score(y_binary, y_pred_cv)
precision = precision_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store in model library
model_library[model_name] = best_qda

# Store results 
records.append({
    "Model": model_name,
    "Classification": "Binary",
    "Classification Type": "QDA",
    "Hyperparameter 1 Name": "reg_param", 
    "Hyperparameter 1 Value": best_reg_param,
    "Hyperparameter 2 Name": "NA", 
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": str(param_grid['reg_param']),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity,
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[601  58]
 [ 92 317]]


/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


With a ROC AUC of 0.937 QDA slightly out performs LDA overall. The QDA correctly identified 4 more indica strains than LDA at the expense of misclassfiying 4 staiva strains as indica. Percision therefore went up at the cost of recall. 

## Q3: SVC - Support Vector Classifier

In [14]:
model_name = "SVC_Binary"
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_binary, y_binary)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated prediction with best model
y_pred_cv = cross_val_predict(best_svc, X_binary, y_binary, cv=5)
y_proba_cv = cross_val_predict(best_svc, X_binary, y_binary, cv=5, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_binary, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_binary, y_proba_cv)
cv_accuracy = accuracy_score(y_binary, y_pred_cv)
precision = precision_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store in model library
model_library[model_name] = best_svc

# Store results 
records.append({
    "Model": model_name,
    "Classification": "Binary",
    "Classification Type": "SVC",
    "Hyperparameter 1 Name": "C", 
    "Hyperparameter 1 Value": best_C,
    "Hyperparameter 2 Name": "NA", 
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": str(param_grid['C']),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity,
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[603  56]
 [ 87 322]]


The SVC model did well with a ROC AUC of 0.937 however QDA still did better. It was was the best so far at identifying indica with the most correctly identified indicat strains (603) and the least number of indica misclassified as sativa (56). It had a higher accuracy at 0.866 than QDA and SVC. 

## Q4: SVM - Support Vector Machine

In [15]:
model_name = "SVM_Binary"
svm_model = SVC(kernel='poly', probability=True, random_state=67)

# Parameter grid for SVM 
param_grid = {
    'C': [0.1, 1, 10, 100],
    'degree': [2, 3, 4, 5],
    'coef0': [0, 1]
}

# GridSearchCV
grid_search = GridSearchCV(
    svm_model,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_binary, y_binary)

# Best model
best_svm = grid_search.best_estimator_
best_C = grid_search.best_params_['C']
best_degree = grid_search.best_params_['degree']
best_coef0 = grid_search.best_params_['coef0']

# Cross-validated prediction with best model
y_pred_cv = cross_val_predict(best_svm, X_binary, y_binary, cv=5)
y_proba_cv = cross_val_predict(best_svm, X_binary, y_binary, cv=5, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_binary, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_binary, y_proba_cv)
cv_accuracy = accuracy_score(y_binary, y_pred_cv)
precision = precision_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_binary, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store in model library
model_library[model_name] = best_svm

# Store results 
records.append({
    "Model": model_name,
    "Classification": "Binary",
    "Classification Type": "SVM",
    "Hyperparameter 1 Name": "C", 
    "Hyperparameter 1 Value": best_C,
    "Hyperparameter 2 Name": "degree", 
    "Hyperparameter 2 Value": best_degree,
    "Hyperparameter 3 Name": "coef0",
    "Hyperparameter 3 Value": best_coef0,
    "Range Tested": str(param_grid),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity,
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[610  49]
 [100 309]]


SVM is the best model yet with a ROC AUC of 0.939. It however did not have teh highest overall accuracy (though only 0.06 less than SVC). This one had the best performance of all models at identifying indica however this came at tegh cost of having the highest false negatives, 100 sativa strains were misclassified as indica. 

In [16]:
dfPt1 = pd.DataFrame(records)
dfPt1.sort_values('ROC AUC', ascending=False)

,Model,Classification,Classification Type,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested,ROC AUC,CV Accuracy,Confusion Matrix,Precision,Recall,Specificity
3,SVM_Binary,Binary,SVM,C,0.1,degree,3,coef0,1,"{'C': [0.1, 1, 10, 100], 'degree': [2, 3, 4, 5...",0.939039,0.860487,"[[610, 49], [100, 309]]",0.863128,0.755501,0.925645
1,QDA_Binary,Binary,QDA,reg_param,0.2,NA,NA,NA,NA,"[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ...",0.937465,0.859551,"[[601, 58], [92, 317]]",0.845333,0.775061,0.911988
2,SVC_Binary,Binary,SVC,C,0.1,NA,NA,NA,NA,"[0.001, 0.01, 0.1, 1, 10, 100, 1000]",0.937117,0.866105,"[[603, 56], [87, 322]]",0.851852,0.787286,0.915023
0,LDA_Binary,Binary,LDA,NA,NA,NA,NA,NA,NA,NA,0.931704,0.859551,"[[597, 62], [88, 321]]",0.838120,0.784841,0.905918


# Part Two: Natural Multiclass
Now use the full dataset, including the Hybrid strains.

In [17]:
# Multiclass Split 
X_multiclass = X
y_multiclass = y

## Q1
Fit a decision tree, plot the final fit, and interpret the results.
Your answer here

In [18]:
model_name = "DecisionTree_Multiclass"
dt_model = DecisionTreeClassifier(random_state=67)

# Parameter grid for Decision Tree
param_grid = {
    'max_depth': [3, 4, 5, 6, 7, 10, 15, 20, None]
}

# GridSearchCV
grid_search = GridSearchCV(
    dt_model,
    param_grid,
    cv=5,
    scoring='roc_auc_ovr', 
    n_jobs=-1
)
grid_search.fit(X_multiclass, y_multiclass)

# Best model
best_dt = grid_search.best_estimator_
best_max_depth = grid_search.best_params_['max_depth']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_dt, X_multiclass, y_multiclass, cv=5)

# Metrics
conf_matrix = confusion_matrix(y_multiclass, y_pred_cv)
cv_accuracy = accuracy_score(y_multiclass, y_pred_cv)

# Probability predictions
y_proba_cv = cross_val_predict(best_dt, X_multiclass, y_multiclass, cv=5, method='predict_proba')
cv_roc_auc = roc_auc_score(y_multiclass, y_proba_cv, multi_class='ovr', average='macro')

# Store in model library
model_library[model_name] = best_dt

# Store results
records.append({
    "Model": model_name,
    "Classification": "Multiclass",
    "Classification Type": "Decision Tree",
    "Hyperparameter 1 Name": "max_depth",
    "Hyperparameter 1 Value": best_max_depth,
    "Hyperparameter 2 Name": "NA",
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": str(param_grid['max_depth']),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": "NA",
    "Recall": "NA",
    "Specificity": "NA"
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[776 215 132]
 [234 415  10]
 [221  21 167]]


Despite all the depths tested, 3 was the best, this produced the lowest Roc AUC we have seen so far 0.746 with a accuracy of 61.98%. Of all the strains it was the best at classifying indica. It correctly classified: 415 (62.9% of actual indicas.

## Q2
Repeat the analyses from Part One for LDA, QDA, and KNN.

### LDA Multiclass

In [19]:
model_name = "LDA_Multiclass"
lda_model = LinearDiscriminantAnalysis()

# Cross-validated predictions
y_pred_cv = cross_val_predict(lda_model, X_multiclass, y_multiclass, cv=5)

# Metrics
conf_matrix = confusion_matrix(y_multiclass, y_pred_cv)
cv_accuracy = accuracy_score(y_multiclass, y_pred_cv)

# Metrics
y_proba_cv = cross_val_predict(lda_model, X_multiclass, y_multiclass, cv=5, method='predict_proba')
cv_roc_auc = roc_auc_score(y_multiclass, y_proba_cv, multi_class='ovr', average='macro')

# Fit
lda_model.fit(X_multiclass, y_multiclass)

# Store in model library
model_library[model_name] = lda_model

# Store results
records.append({
    "Model": model_name,
    "Classification": "Multiclass",
    "Classification Type": "LDA",
    "Hyperparameter 1 Name": "NA",
    "Hyperparameter 1 Value": "NA",
    "Hyperparameter 2 Name": "NA",
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": "NA",
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": "NA",
    "Recall": "NA",
    "Specificity": "NA"
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[750 222 151]
 [195 451  13]
 [213  21 175]]


With a ROC AUC of 0.772 it is the second-highest among multiclass models, outperforming Decision Tree (0.746) and KNN (0.756) but trailing QDA (0.776). However, it correstly classified indica more often than decision tree 68.4%. 

### QDA Multiclass

In [20]:
model_name = "QDA_Multiclass"
qda_model = QuadraticDiscriminantAnalysis()

# Parameter grid for QDA
param_grid = {
    'reg_param': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

# GridSearchCV
grid_search = GridSearchCV(
    qda_model,
    param_grid,
    cv=5,
    scoring='roc_auc_ovr',
    n_jobs=-1
)
grid_search.fit(X_multiclass, y_multiclass)

# Best model
best_qda = grid_search.best_estimator_
best_reg_param = grid_search.best_params_['reg_param']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_qda, X_multiclass, y_multiclass, cv=5)

# Metrics
conf_matrix = confusion_matrix(y_multiclass, y_pred_cv)
cv_accuracy = accuracy_score(y_multiclass, y_pred_cv)

# Metrics 
y_proba_cv = cross_val_predict(best_qda, X_multiclass, y_multiclass, cv=5, method='predict_proba')
cv_roc_auc = roc_auc_score(y_multiclass, y_proba_cv, multi_class='ovr', average='macro')

# Store in model library
model_library[model_name] = best_qda

# Store results
records.append({
    "Model": model_name,
    "Classification": "Multiclass",
    "Classification Type": "QDA",
    "Hyperparameter 1 Name": "reg_param",
    "Hyperparameter 1 Value": best_reg_param,
    "Hyperparameter 2 Name": "NA",
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": str(param_grid['reg_param']),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": "NA",
    "Recall": "NA",
    "Specificity": "NA"
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


Confusion Matrix (CV):
[[943 167  13]
 [262 396   1]
 [381  10  18]]


The Multiclass QDA used the regulation parameter of 0.5 instead of binary's 0.2. Its performance was also more disappointing with a ROC AUC of 0.776 and an accuracy of 0.619. It was notly terrible at classifying satiuva strains, correctly classiflying only 18 of them, instead predicting most sativa as hybrids, hybrids being the only strain it predicted well. 

### KNN Multiclass

In [21]:
model_name = "KNN_Multiclass"
knn_model = KNeighborsClassifier()

# Parameter grid for KNN
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
}

# GridSearchCV
grid_search = GridSearchCV(
    knn_model,
    param_grid,
    cv=5,
    scoring='roc_auc_ovr',
    n_jobs=-1
)
grid_search.fit(X_multiclass, y_multiclass)

# Best model
best_knn = grid_search.best_estimator_
best_k = grid_search.best_params_['n_neighbors']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_knn, X_multiclass, y_multiclass, cv=5)

# Metrics
conf_matrix = confusion_matrix(y_multiclass, y_pred_cv)
cv_accuracy = accuracy_score(y_multiclass, y_pred_cv)
y_proba_cv = cross_val_predict(best_knn, X_multiclass, y_multiclass, cv=5, method='predict_proba')
cv_roc_auc = roc_auc_score(y_multiclass, y_proba_cv, multi_class='ovr', average='macro')

# Store in model library
model_library[model_name] = best_knn

# Store results
records.append({
    "Model": model_name,
    "Classification": "Multiclass",
    "Classification Type": "KNN",
    "Hyperparameter 1 Name": "KNNeighbors",
    "Hyperparameter 1 Value": best_k,
    "Hyperparameter 2 Name": "NA",
    "Hyperparameter 2 Value": "NA",
    "Hyperparameter 3 Name": "NA", 
    "Hyperparameter 3 Value": "NA",
    "Range Tested": str(param_grid['n_neighbors']),
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": "NA",
    "Recall": "NA",
    "Specificity": "NA"
})

# Print
print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[844 211  68]
 [263 392   4]
 [304  21  84]]


With a Nearest Neighbors of 21, KNN sits in the middle of the multiuclass pack with a ROC AUC of 0.756 and a accuracy of .6025. Its not AS bad at classifying sativa as QDA with 84 correctly classified (20.5%) but that is still unacceptably low.

## Q3
Were your metrics better or worse than in Part One? Why? Which categories were most likely to get mixed up, according to the confusion matrices? Why?

In [22]:
dfPt2 = pd.DataFrame(records)
dfPt2.sort_values('ROC AUC', ascending=False)

,Model,Classification,Classification Type,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested,ROC AUC,CV Accuracy,Confusion Matrix,Precision,Recall,Specificity
3,SVM_Binary,Binary,SVM,C,0.1,degree,3,coef0,1,"{'C': [0.1, 1, 10, 100], 'degree': [2, 3, 4, 5...",0.939039,0.860487,"[[610, 49], [100, 309]]",0.863128,0.755501,0.925645
1,QDA_Binary,Binary,QDA,reg_param,0.2,NA,NA,NA,NA,"[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ...",0.937465,0.859551,"[[601, 58], [92, 317]]",0.845333,0.775061,0.911988
2,SVC_Binary,Binary,SVC,C,0.1,NA,NA,NA,NA,"[0.001, 0.01, 0.1, 1, 10, 100, 1000]",0.937117,0.866105,"[[603, 56], [87, 322]]",0.851852,0.787286,0.915023
0,LDA_Binary,Binary,LDA,NA,NA,NA,NA,NA,NA,NA,0.931704,0.859551,"[[597, 62], [88, 321]]",0.83812,0.784841,0.905918
6,QDA_Multiclass,Multiclass,QDA,reg_param,0.5,NA,NA,NA,NA,"[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ...",0.776469,0.619352,"[[943, 167, 13], [262, 396, 1], [381, 10, 18]]",NA,NA,NA
5,LDA_Multiclass,Multiclass,LDA,NA,NA,NA,NA,NA,NA,NA,0.771512,0.628024,"[[750, 222, 151], [195, 451, 13], [213, 21, 175]]",NA,NA,NA
7,KNN_Multiclass,Multiclass,KNN,KNNeighbors,21,NA,NA,NA,NA,"[3, 5, 7, 9, 11, 13, 15, 17, 19, 21]",0.756017,0.602465,"[[844, 211, 68], [263, 392, 4], [304, 21, 84]]",NA,NA,NA
4,DecisionTree_Multiclass,Multiclass,Decision Tree,max_depth,3,NA,NA,NA,NA,"[3, 4, 5, 6, 7, 10, 15, 20, None]",0.745767,0.619808,"[[776, 215, 132], [234, 415, 10], [221, 21, 167]]",NA,NA,NA


The multiclass models are notably worse than the binary models. This is to be expected as hybrid strains are somewhere in the middle of sativa and indica by design, making them harder to classify. This would of course lower the corresponding Accuracy and ROC AUCs by confusing the models. We saw this with the rampant missclassification of Sativa strains as hybrid. 

# Part Three: Multiclass from Binary
Consider two models designed for binary classification: SVC and Logistic Regression.

## Q1
Fit and report metrics for OvR versions of the models. That is, for each of the two model types, create three models:

    - Indica vs. Not Indica
    - Sativa vs. Not Sativa
    - Hybrid vs. Not Hybrid

In [23]:
records_part3 = []

In [24]:
model_name = "SVC_OvR_Indica"
y_ovr = (y_multiclass == 'indica').astype(int)

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "SVC",
    "Comparison": "Indica vs. Not Indica",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1286  246]
 [ 211  448]]


In [25]:
model_name = "SVC_OvR_Sativa"
y_ovr = (y_multiclass == 'sativa').astype(int)

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "SVC",
    "Comparison": "Sativa vs. Not Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1777    5]
 [ 409    0]]


In [26]:
model_name = "SVC_OvR_Hybrid"
y_ovr = (y_multiclass == 'hybrid').astype(int)

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "SVC",
    "Comparison": "Hybrid vs. Not Hybrid",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[515 553]
 [289 834]]


In [27]:
model_name = "Logistic_OvR_Indica"
y_ovr = (y_multiclass == 'indica').astype(int)

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "Logistic Regression",
    "Comparison": "Indica vs. Not Indica",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1358  174]
 [ 258  401]]


In [28]:
model_name = "Logistic_OvR_Sativa"
y_ovr = (y_multiclass == 'sativa').astype(int)

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "Logistic Regression",
    "Comparison": "Sativa vs. Not Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1782    0]
 [ 408    1]]


In [29]:
model_name = "Logistic_OvR_Hybrid"
y_ovr = (y_multiclass == 'hybrid').astype(int)

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_multiclass, y_ovr)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_multiclass, y_ovr, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovr, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovr, y_proba_cv)
cv_accuracy = accuracy_score(y_ovr, y_pred_cv)
precision = precision_score(y_ovr, y_pred_cv, zero_division=0)
recall = recall_score(y_ovr, y_pred_cv, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvR",
    "Model Type": "Logistic Regression",
    "Comparison": "Hybrid vs. Not Hybrid",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[599 469]
 [371 752]]


## Q2
Which of the six models did the best job distinguishing the target category from the rest? Which did the worst? Does this make intuitive sense?

In [30]:
dfPt3_OvR = pd.DataFrame(records_part3)
dfPt3_OvR[dfPt3_OvR['Classification'] == 'OvR'].sort_values('ROC AUC', ascending=False)

,Model,Classification,Model Type,Comparison,Best C,ROC AUC,CV Accuracy,Confusion Matrix,Precision,Recall,Specificity
3,Logistic_OvR_Indica,OvR,Logistic Regression,Indica vs. Not Indica,0.10,0.842796,0.802830,"[[1358, 174], [258, 401]]",0.697391,0.608498,0.886423
0,SVC_OvR_Indica,OvR,SVC,Indica vs. Not Indica,0.01,0.836705,0.791419,"[[1286, 246], [211, 448]]",0.645533,0.679818,0.839426
4,Logistic_OvR_Sativa,OvR,Logistic Regression,Sativa vs. Not Sativa,0.01,0.816081,0.813784,"[[1782, 0], [408, 1]]",1.000000,0.002445,1.000000
1,SVC_OvR_Sativa,OvR,SVC,Sativa vs. Not Sativa,0.10,0.755212,0.811045,"[[1777, 5], [409, 0]]",0.000000,0.000000,0.997194
5,Logistic_OvR_Hybrid,OvR,Logistic Regression,Hybrid vs. Not Hybrid,0.10,0.659413,0.616613,"[[599, 469], [371, 752]]",0.615889,0.669635,0.560861
2,SVC_OvR_Hybrid,OvR,SVC,Hybrid vs. Not Hybrid,100.00,0.614227,0.615701,"[[515, 553], [289, 834]]",0.601298,0.742654,0.482210


Logistic for Indica was the model that performed better in ROC AUC score (0.843) out of all of the models. As we saw in earlier models, we are best at identifying indica and worst at identifying hybrids. Overall the Logistic models did better than the SVC models which is unexpected since SVC is supposed to be better at handling more features than Logistic. For example, Logistic outperformed SVC for Indica (0.843 vs 0.837), Sativa (0.816 vs 0.755), and Hybrid (0.659 vs 0.614). Our worst model is SVC_OvR_Hybrid (0.614 ROC-AUC). I suspect it's easier to distinguish an indica from everything else which is why it constantly outperforms, while hybrids share features with both parent types making them much harder to separate from the "Not Hybrid" group.

## Q3
Fit and report metrics for OvO versions of the models. That is, for each of the two model types, create three models:

    - Indica vs. Sativa
    - Indica vs. Hybrid
    - Hybrid vs. Sativa

In [31]:
model_name = "SVC_OvO_Indica_Sativa"

# Filter 
mask_ovo = y_multiclass.isin(['indica', 'sativa'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "SVC",
    "Comparison": "Indica vs. Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[620  39]
 [115 294]]


In [32]:
model_name = "SVC_OvO_Indica_Hybrid"

# Filter 
mask_ovo = y_multiclass.isin(['indica', 'hybrid'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='hybrid', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='hybrid', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "SVC",
    "Comparison": "Indica vs. Hybrid",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[890 233]
 [208 451]]


In [33]:
model_name = "SVC_OvO_Hybrid_Sativa"

# Filter 
mask_ovo = y_multiclass.isin(['hybrid', 'sativa'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# SVC model
svc_model = SVC(kernel='linear', probability=True, random_state=67)

# Parameter grid for SVC
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    svc_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_svc = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_svc, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "SVC",
    "Comparison": "Hybrid vs. Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1123    0]
 [ 409    0]]


In [34]:
model_name = "Logistic_OvO_Indica_Sativa"

# Filter 
mask_ovo = y_multiclass.isin(['indica', 'sativa'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "Logistic Regression",
    "Comparison": "Indica vs. Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[607  52]
 [ 86 323]]


In [35]:
model_name = "Logistic_OvO_Indica_Hybrid"

# Filter 
mask_ovo = y_multiclass.isin(['indica', 'hybrid'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='hybrid', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='hybrid', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "Logistic Regression",
    "Comparison": "Indica vs. Hybrid",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[936 187]
 [238 421]]


In [36]:
model_name = "Logistic_OvO_Hybrid_Sativa"

# Filter 
mask_ovo = y_multiclass.isin(['hybrid', 'sativa'])
X_ovo = X_multiclass[mask_ovo]
y_ovo = y_multiclass[mask_ovo]

# Logistic Regression model
logreg_model = LogisticRegression(random_state=67, max_iter=1000)

# Parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid_search = GridSearchCV(
    logreg_model,
    param_grid,
    cv=3, 
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(X_ovo, y_ovo)

# Best model
best_logreg = grid_search.best_estimator_
best_C = grid_search.best_params_['C']

# Cross-validated predictions
y_pred_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3)
y_proba_cv = cross_val_predict(best_logreg, X_ovo, y_ovo, cv=3, method='predict_proba')[:, 1]

# Metrics
conf_matrix = confusion_matrix(y_ovo, y_pred_cv)
tn, fp, fn, tp = conf_matrix.ravel()
cv_roc_auc = roc_auc_score(y_ovo, y_proba_cv)
cv_accuracy = accuracy_score(y_ovo, y_pred_cv)
precision = precision_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
recall = recall_score(y_ovo, y_pred_cv, pos_label='sativa', zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Store results in records
records_part3.append({
    "Model": model_name,
    "Classification": "OvO",
    "Model Type": "Logistic Regression",
    "Comparison": "Hybrid vs. Sativa",
    "Best C": best_C,
    "ROC AUC": cv_roc_auc,
    "CV Accuracy": cv_accuracy,
    "Confusion Matrix": conf_matrix,
    "Precision": precision,
    "Recall": recall,
    "Specificity": specificity
})

print("Confusion Matrix (CV):")
print(conf_matrix)

Confusion Matrix (CV):
[[1123    0]
 [ 409    0]]


## Q4
Which of the six models did the best job distinguishing at differentiating the two groups? Which did the worst? Does this make intuitive sense?

In [37]:
dfPt4_OvR = pd.DataFrame(records_part3)
dfPt4_OvR[dfPt4_OvR['Classification'] == 'OvO'].sort_values('ROC AUC', ascending=False)

,Model,Classification,Model Type,Comparison,Best C,ROC AUC,CV Accuracy,Confusion Matrix,Precision,Recall,Specificity
9,Logistic_OvO_Indica_Sativa,OvO,Logistic Regression,Indica vs. Sativa,0.10,0.937330,0.870787,"[[607, 52], [86, 323]]",0.861333,0.789731,0.921093
6,SVC_OvO_Indica_Sativa,OvO,SVC,Indica vs. Sativa,0.01,0.936191,0.855805,"[[620, 39], [115, 294]]",0.882883,0.718826,0.940819
10,Logistic_OvO_Indica_Hybrid,OvO,Logistic Regression,Indica vs. Hybrid,0.10,0.806587,0.761504,"[[936, 187], [238, 421]]",0.797274,0.833482,0.833482
7,SVC_OvO_Indica_Hybrid,OvO,SVC,Indica vs. Hybrid,0.01,0.801623,0.752525,"[[890, 233], [208, 451]]",0.810565,0.792520,0.792520
11,Logistic_OvO_Hybrid_Sativa,OvO,Logistic Regression,Hybrid vs. Sativa,0.01,0.745762,0.733029,"[[1123, 0], [409, 0]]",0.000000,0.000000,1.000000
8,SVC_OvO_Hybrid_Sativa,OvO,SVC,Hybrid vs. Sativa,0.01,0.707417,0.733029,"[[1123, 0], [409, 0]]",0.000000,0.000000,1.000000


It seems that once again Logistic outperforms SVC unexpectedly. The Indica vs. Sativa split was the easiest for the models to handle with Logistic achieving 0.937 ROC-AUC and SVC achieving 0.936. Performance goes down from there with Indica vs. Hybrid (Logistic: 0.807, SVC: 0.802) and is worst for Hybrid vs. Sativa (Logistic: 0.746, SVC: 0.707). 

The performance of Hybrid vs. Sativa being the worst for both models lines up with my theory that there are not as many meaningful differences between Sativa and Hybrid in comparison to Indica to both. Both models completely failed to predict any Sativa cases in the Hybrid vs. Sativa comparison (0% recall), predicted everything as Hybrid.

## Q5
Suppose you had simply input the full data, with three classes, into the LogisticRegression function. Would this have automatically taken an "OvO" approach or an "OvR" approach?

LogisticRegression automatically uses OvR. It creates three binary classifiers: one for Indica vs. Not Indica, etc.

What about for SVC?

SVC automatically uses OvO. SVC trains a binary classifier for every pair of classes: Indica vs. Sativa, Indica vs. Hybrid, and Hybrid vs. Sativa.

Note: You do not actually have to run code here - you only need to look at sklearn's documentation to see how these functions handle multiclass input.